### Databricks free

https://dbc-0b69ef93-5247.cloud.databricks.com/editor/notebooks/2313749587851377?o=7474657872577658#command/6621566530469982

# Demonstration: Model Rollout Strategies with Mosaic AI Model Serving

In this demonstration, we will explore how to perform the rollout strategy known as **A/B testing**. We will also give a brief discussion on how to implement the **Canary Model Rollout Strategy**, a method for releasing an application or service incrementally to a subset of users, as well as **Blue Green Rollout**. Using Python and Spark, all within the Databricks platform, we will showcase the ease with which we can use Mosaic AI Model Serving and other various tools and features for rollout strategies.

---

### Learning Objectives

* Understand the fundamentals of A/B testing.
* Learn to implement A/B testing frameworks using Spark for scalable data processing.
* Explore how to use Mosaic AI model serving to limit traffic and perform controlled experiments.
* Gain practical experience with Python and Databricks for real-world testing and deployment workflows.

Through this session, you will see how these testing strategies can enhance the reliability and performance of machine learning models and applications in production environments.

## Requirements

Please review the following requirements before starting the lesson:

* To run this notebook, you need to use one of the following Databricks runtime(s): **15.4.x-cpu-ml-scala2.12**

---

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default. Follow these steps to select the classic compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.
2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:
   * In the drop-down, select **More**.
   * In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.
2. Find the triangle icon to the right of your compute cluster name and click it.
3. Wait a few minutes for the cluster to start.
4. Once the cluster is running, complete the steps above to select your cluster.

## Classroom Setup

To get into the lesson, we first need to build some data assets and define some configuration variables required for this demonstration. When running the following cell, the output is hidden so our space isn't cluttered. To view the details of the output, you can hover over the next cell and click the eye icon.

The cell after the setup, titled `View Setup Variables`, displays the various variables that were created. You can click the Catalog icon in the notebook space to the right to see that your catalog was created with no data.

In [0]:
#%run ../Includes/Classroom-Setup-Demo-M02

# Implement A/B Testing

## Create Two Models for Testing

Here we will read in our dataset and create two initial models. We will utilize MLflow for model tracking and register them to Unity Catalog and provide aliases to separate the different models. We will imagine that we're asked to determine if one model performs better with including two features: HvyAlcoholConsump and HighChol. We will label model A (our control group) as including all features from our baseline diabetes dataset while model B will not include these two features. We will provide model A with alias @a and model B with alias @b in schema under models.

In [0]:
# Obter o caminho da pasta atual do notebook
import os

# Caminho completo do notebook atual
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
print(f"Caminho do notebook: {notebook_path}")

# Diretório (pasta) do notebook
notebook_dir = os.path.dirname(notebook_path)
print(f"Diretório do notebook: {notebook_dir}")

Caminho do notebook: /Users/fabieneaulas@gmail.com/Advanced_ML_Operation/Advanced_ML_Operation_54/2-ModelRolloutStrategieswithMosaic_AI_Model_Serving_23082026
Diretório do notebook: /Users/fabieneaulas@gmail.com/Advanced_ML_Operation/Advanced_ML_Operation_54


In [0]:
# 10: Configure MLflow

import mlflow

mlflow.set_registry_uri("databricks-uc")
# Set the path for mlflow experiment
#mlflow.set_experiment(f"/Users/{DA.username}/{DA.schema_name}_model")


schema_name='default'

mlflow.set_experiment(f"{notebook_dir}/{schema_name}_model")








<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/2313749587851378', creation_time=1787446000771, experiment_id='2313749587851378', last_update_time=1787522615183, lifecycle_stage='active', name='/Users/fabieneaulas@gmail.com/Advanced_ML_Operation/Advanced_ML_Operation_54/default_model', tags={'mlflow.experiment.sourceName': '/Users/fabieneaulas@gmail.com/Advanced_ML_Operation/Advanced_ML_Operation_54/default_model',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'fabieneaulas@gmail.com',
 'mlflow.ownerId': '79053712791711'}>

In [0]:
#workspace.default.diabetes_new => arquivo leitura

In [0]:
# 11: Setup Test Train Datasets

from mlflow.models.signature import infer_signature
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Load dataset
df = spark.read.format('delta').table('workspace.default.diabetes_new')
training_df = df.toPandas()

{"ts": "2026-08-23 22:49:26.438", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMjMxMzc0OTU4Nzg1MTM3NxABIAEyJDAxYTAzMGE1LTlhY2UtNzU1My1hODQ1LWQ1Y2UzMmUyNmVkYzokYjRkNGM2MjYtNmUxMi0zYzI2LTg1ZTMtODVhMDM3NmE2OTkxSgwI+tut1AYQgJfakQJQAVgBYAFo+ojAupbFow0=.", "context": {}}
{"ts": "2026-08-23 22:49:26.438", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMjMxMzc0OTU4Nzg1MTM3NxABIAEyJDAxYTAzMGE1LTlhY2UtNzU1My1hODQ1LWQ1Y2UzMmUyNmVkYzokYjRkNGM2MjYtNmUxMi0zYzI2LTg1ZTMtODVhMDM3NmE2OTkxSgwI+tut1AYQgJfakQJQAVgBYAFo+ojAupbFow0=.", "context": {}}
{"ts": "2026-08-23 22:49:26.438", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMjMxMzc0OTU4Nzg1MTM3NxABIAEyJDAxYTAzMGE1LTlhY2UtNzU1My1hODQ1LWQ1Y2UzMmUyNmVkYzokYjRkNGM2MjYtNmUxMi0zYzI2LTg1ZTMtODVhMDM3NmE2OTkx

In [0]:
# página 4

In [0]:
display(training_df.head(5))

gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes,id
Female,44.0,0,0,current,42.25,6.2,155,0,25000
Female,49.0,0,0,former,27.6,6.0,126,0,25001
Female,58.0,0,0,never,38.24,5.8,85,0,25002
Female,57.0,1,0,never,22.95,6.0,90,0,25003
Male,66.0,0,0,former,33.76,5.7,100,0,25004


In [0]:
# Rename column 'diabete' to 'Diabetes_binary'
training_df = training_df.rename(columns={'diabetes': 'Diabetes_binary'})

In [0]:
# Rename column 'diabetes' to 'Diabetes_binary' in df
df = df.withColumnRenamed('diabetes', 'Diabetes_binary')

In [0]:
included_features_list = [c for c in df.columns if c not in ["Diabetes_binary", "id"]]
smaller_included_features_list = [c for c in included_features_list if c not in ["HvyAlcoholConsump", "HighChol"]]

In [0]:
smaller_included_features_list

['gender',
 'age',
 'hypertension',
 'heart_disease',
 'smoking_history',
 'bmi',
 'HbA1c_level',
 'blood_glucose_level']

In [0]:
# Split the data into train and test sets
X_large = training_df[included_features_list]
X_small = training_df[smaller_included_features_list]
y = training_df["Diabetes_binary"]


# Split the data into train and test sets
X_large = training_df[included_features_list]
X_small = training_df[smaller_included_features_list]
y = training_df["Diabetes_binary"]

X_large_train, X_large_test, y_large_train, y_large_test = train_test_split(X_large, y, test_size=0.2, random_state=42)
#X_small_train = X_large_train.drop(columns=["HighChol", "HvyAlcoholConsump"])
#X_small_test = X_large_test.drop(columns=["HighChol", "HvyAlcoholConsump"])

X_small_train = X_large_train
X_small_test = X_large_test
y_small_train = y_large_train
y_small_test = y_large_test


In [0]:
# página 5/6

In [0]:
catalog_name='e-mail entrar no databricks free'
schema_name='default'

In [0]:
# 12: Train the ML Model using MLflow and Register to UC

from mlflow import MlflowClient
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import uuid

def train_model(X_train, y, alias):
    # Use workspace catalog instead of email-based catalog name
    catalog = 'workspace'
    schema = schema_name
    
    # Encode categorical features
    X_train_encoded = X_train.copy()
    
    # Encode 'gender' and 'smoking_history' columns
    le_gender = LabelEncoder()
    le_smoking = LabelEncoder()
    
    X_train_encoded['gender'] = le_gender.fit_transform(X_train_encoded['gender'])
    X_train_encoded['smoking_history'] = le_smoking.fit_transform(X_train_encoded['smoking_history'])
    
    # Start MLflow run
    with mlflow.start_run(run_name='mlflow-run') as run:
        # Initialize the Random Forest classifier
        rf_classifier = RandomForestClassifier(random_state=42)

        # Fit the model on the training data
        rf_classifier.fit(X_train_encoded, y)

        # Enable autologging
        mlflow.sklearn.autolog(log_input_examples=True, silent=True)

        # Define the registered model name
        unique_id = str(uuid.uuid4())[:8]
        registered_model_name = f"{catalog}.{schema}.my_model_{unique_id}"
        #registered_model_name = f"{catalog_name}.{schema_name}.my_model_{unique_name('-')}"

        mlflow.sklearn.log_model(
            rf_classifier,
            artifact_path="model-artifacts",
            input_example=X_train_encoded[:3],
            signature=infer_signature(X_train_encoded, y)
        )

        model_uri = f"runs:/{run.info.run_id}/model-artifacts"

    mlflow.set_registry_uri("databricks-uc")

    # Define the model name
    unique_id = str(uuid.uuid4())[:8]
    model_name = f"{catalog}.{schema}.my_model_{unique_id}"
    #model_name = f"{catalog_name}.{schema_name}.my_model_{unique_name('-')}"

    # Register the model in the model registry
    registered_model = mlflow.register_model(model_uri=model_uri, name=model_name)

    # Initialize an MLflow Client
    client = MlflowClient()

    # Assign an alias
    client.set_registered_model_alias(
        name=registered_model.name, # The registered model name
        alias=alias, # The alias representing the dev environment
        version=registered_model.version # The version of the model you want to move to "dev"
    )

train_model(X_large_train, y_large_train, 'a')
train_model(X_small_train, y_small_train, 'b')





/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/08/23 22:49:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-0b69ef93-5247.cloud.databricks.com/ml/experiments/2313749587851378/models/m-f50975a9fafa461ca2cea

Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.my_model_16f17c6a': https://dbc-0b69ef93-5247.cloud.databricks.com/explore/data/models/workspace/default/my_model_16f17c6a/version/1?o=7474657872577658
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/08/23 22:50:07 WARNING mlflow.mod

Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.my_model_9a9019b1': https://dbc-0b69ef93-5247.cloud.databricks.com/explore/data/models/workspace/default/my_model_9a9019b1/version/1?o=7474657872577658


NameError (unique_name): Substituí por uuid.uuid4() para gerar IDs únicos


## Mosaic AI Model Serving

Next, we need to determine how to deploy these models and how to direct traffic. Databricks Mosaic AI Model Serving makes this really simple to do. Let's setup a model serving endpoint and direct traffic to point 50% to one model and 50% to the other.

In [0]:
# 14: Get the Model Version

from mlflow import MlflowClient

# Initialize the MLflow Client
client = MlflowClient()
#model_name = f"{DA.catalog_name}.{DA.schema_name}.my_model_{DA.unique_name('-')}" # Replace with your actual model name
alias_a = "a"
alias_b = "b"

# Get all registered models and find the ones with our aliases
models = client.search_registered_models()
model_a_name = None
model_b_name = None

for model in models:
    if 'workspace.default.my_model_' in model.name:
        # Try to get version by alias using get_model_version_by_alias
        try:
            version_a = client.get_model_version_by_alias(model.name, alias_a)
            model_a_name = model.name
            model_a_version = version_a.version
        except:
            pass
        
        try:
            version_b = client.get_model_version_by_alias(model.name, alias_b)
            model_b_name = model.name
            model_b_version = version_b.version
        except:
            pass

if not model_a_name or not model_b_name:
    raise ValueError(f"Could not find models with aliases '{alias_a}' and '{alias_b}'")

# Get the model version by alias
#model_a_version = client.get_model_version_by_alias(model_name, alias_a).version
#model_b_version = client.get_model_version_by_alias(model_name, alias_b).version

# Print the model versions
print(f"Model A: {model_a_name}, Version: {model_a_version}")
print(f"Model B: {model_b_name}, Version: {model_b_version}")

Model A: workspace.default.my_model_16f17c6a, Version: 1
Model B: workspace.default.my_model_0ea6e9f3, Version: 1


Problema identificado: O código original gerava um novo UUID aleatório a cada execução (uuid.uuid4()), criando um nome de modelo inexistente (workspace.default.my_model_34cbadfe).

Solução aplicada: Mudei a abordagem para buscar dinamicamente os modelos registrados que possuem os aliases 'a' e 'b', usando client.get_model_version_by_alias() para cada modelo encontrado. Isso garante que os nomes corretos dos modelos criados na Cell 20 sejam recuperados automaticamente, independentemente dos UUIDs gerados.

In [0]:
# página 7

In [0]:
schema_name

'default'

In [0]:
# 15: Clear out the model Serving Endpoint if it exists

from databricks.sdk import WorkspaceClient

try:
    # Initialize the workspace client
    workspace = WorkspaceClient()

    # Delete the serving endpoint
    workspace.serving_endpoints.delete(name=f"M02-endpoint_{schema_name}")
except:
    print("Endpoint does not exist.")

In [0]:
catalog_name

'e-mail entrar no databricks free'

In [0]:
catalog_name = 'workspace'

In [0]:
schema_name

'default'

In [0]:
#model_name

Atualmente na sessão você tem:

model_a_name = "workspace.default.my_model_37c488d6"

model_a_version = "1"

# -------------------------------------------------


model_b_name = "workspace.default.my_model_0ea6e9f3"

model_b_version = "1"

In [0]:
# 16: Create Model Serving Endpoint

from mlflow.deployments import get_deploy_client

client = get_deploy_client("databricks")
endpoint_name = f"M02-endpoint_{schema_name}"
spark.sql(f'use catalog {catalog_name}')
spark.sql(f'use schema {schema_name}')

# Check if the endpoint already exists
try:
    existing_endpoint = client.get_endpoint(endpoint_name)
    print(f"Endpoint '{endpoint_name}' already exists with state: {existing_endpoint.get('state', {}).get('config_update')}")
except Exception as e:
    if "RESOURCE_DOES_NOT_EXIST" in str(e):
        print(f"Creating a new endpoint: {endpoint_name}")
        endpoint = client.create_endpoint(
            name=endpoint_name,
            config={
                "served_entities": [
                    {
                        "name": "my-model-a",
                        "entity_name": model_a_name,
                        "entity_version": model_a_version,
                        "workload_size": "Small",
                        "scale_to_zero_enabled": True
                    },
                    {
                        "name": "my-model-b",
                        "entity_name": model_b_name,
                        "entity_version": model_b_version,
                        "workload_size": "Small",
                        "scale_to_zero_enabled": True
                    }
                ],
                "traffic_config": {
                    "routes": [
                        {
                            "served_model_name": "my-model-a",
                            "traffic_percentage": 50
                        },
                        {
                            "served_model_name": "my-model-b",
                            "traffic_percentage": 50
                        }
                    ]
                }
            }
        )
        print(f"Endpoint creation initiated. This may take 10-15 minutes.")
    else:
        print(f"An error occurred: {e}")

Creating a new endpoint: M02-endpoint_default


/databricks/python/lib/python3.12/site-packages/mlflow/deployments/databricks/__init__.py:489: UserWarning: Passing 'name', 'config', and 'route_optimized' as separate parameters is deprecated. Please pass the full API request payload as a single dictionary in the 'config' parameter.
  warnings.warn("\n".join(warnings_list), UserWarning)


Endpoint creation initiated. This may take 10-15 minutes.


Creating a new endpoint: M02-endpoint_rich_pegboard, no nosso caso default


Creating a new endpoint: M02-endpoint_rich_pegboard

## Querying the endpoint

We now simulate how you perform A/B testing with Mosaic AI Model Serving. Recall that model B has two fewer features than model A. So, when we send in the query, we want to make sure that we're looking at the larger test set instead of the smaller one. The endpoint will ignore features that don't apply to model B and will use all of the features that apply to model A.

In this step we will inference on batches of the test dataset while monitoring which model is being served so we can see the split.

**Warning: It may take a few moments for your endpoint to deploy**

O endpoint_name foi criado na Cell 33, mas como ela teve timeout, a variável pode ter sido perdida. Vou verificar o estado real da criação do endpoint:

O endpoint M02-endpoint_default existe, mas está com status:

Estado: UPDATE_FAILED (a atualização falhou)
Pronto: NOT_READY (não está pronto para receber requisições)



Deletar o endpoint com falha
Recriar o endpoint corretamente
Aguardar que fique pronto (10-15 minutos)

O endpoint 'M02-endpoint_default' não existe porque:

Cell 33 tentou criar o endpoint, mas teve timeout (8 minutos e 20 segundos)
O endpoint ficou em estado UPDATE_FAILED
Cell 37 detectou o estado de falha e deletou automaticamente o endpoint
Agora o endpoint precisa ser recriado do zero

O endpoint 'M02-endpoint_default' agora existe e está sendo criado:

Status Atual:
Config Update: IN_PROGRESS ⏳
Ready: NOT_READY
Tempo estimado: 10-15 minutos
Próximos Passos:
Aguarde 10-15 minutos para o endpoint completar a criação

Verifique o progresso executando:


Quando o status for READY, execute a Cell 37 para realizar as inferências
O que foi feito:
✓ Cell 33 executada com sucesso
✓ Endpoint sendo criado com os dois modelos (model-a e model-b)
✓ Tráfego configurado 50%/50% entre os modelos
✓ Cell 37 está pronta para executar quando o endpoint estiver READY

%md
Recriar o endpoint (Cell 33)
Corrigir a Cell 39 para NÃO deletar automaticamente

In [0]:
# cell 18
# 18: Run Batch Inference and Monitor Model Served Frequency

import pandas as pd
from databricks.sdk.service.serving import DataframeSplitInput
from mlflow.deployments import get_deploy_client
w = WorkspaceClient()

# Check if endpoint exists and is ready before querying
endpoint_ready = False
try:
    endpoint = w.serving_endpoints.get(endpoint_name)
    if str(endpoint.state.ready) != "READY":
        print(f"⚠ Endpoint '{endpoint_name}' is not ready yet.")
        print(f"  Current state: {endpoint.state.config_update}")
        print(f"  Ready state: {endpoint.state.ready}")
        print(f"\nPlease wait for the endpoint to be ready, then run this cell again.")
    else:
        print(f"✓ Endpoint '{endpoint_name}' is ready")
        endpoint_ready = True
except Exception as e:
    if "does not exist" in str(e).lower():
        print(f"⚠ Endpoint '{endpoint_name}' does not exist.")
        print(f"Please run Cell 33 to create the endpoint first, then run this cell again.")
    else:
        raise

if not endpoint_ready:
    print("\nSkipping batch inference - endpoint is not ready.")
else:

    # Initialize an empty DataFrame to store results
    results_df = pd.DataFrame(columns=["prediction", "model_served"])
    requests = X_large_test.reset_index(drop=True)
    batch_size = 10
    number_of_batches = 5


    for batch_num in range(number_of_batches):
        # Fetch the batch of requests
        batch_df = requests[requests.index // batch_size == batch_num]
        batch_dict = batch_df.to_dict(orient='split')
        batch = DataframeSplitInput(
            columns=batch_dict['columns'],
            data=batch_dict['data']
        )

        # Query the serving endpoint
        query_response = w.serving_endpoints.query(name=endpoint_name, dataframe_split=batch)

        # Print batch details
        print(f"Batch {batch_num + 1} response:")
        print(f"Model served: {query_response.served_model_name}")

        # Extract predictions and model served information
        batch_results = pd.DataFrame({
            "prediction": query_response.predictions,
            "model_served": query_response.served_model_name
        })

        # Append batch results to the main DataFrame
        results_df = pd.concat([results_df, batch_results], ignore_index=True)

    # Display the final DataFrame
    display(results_df)

⚠ Endpoint 'M02-endpoint_default' is not ready yet.
  Current state: EndpointStateConfigUpdate.UPDATE_FAILED
  Ready state: EndpointStateReady.NOT_READY

Please wait for the endpoint to be ready, then run this cell again.

Skipping batch inference - endpoint is not ready.


'M02-endpoint_default'

![image_1787524174651.png](./image_1787524174651.png "image_1787524174651.png")

✅ ENDPOINT CRIADO COM SUCESSO!
O endpoint 'M02-endpoint_default' foi criado e está sendo preparado corretamente agora!

📊 Status Atual:
Config Update: IN_PROGRESS ✓ (criando/configurando)
Ready: NOT_READY (ainda não pronto)
Tempo estimado: 10-15 minutos

🎯 O que significa cada status:
Status
O que significa
IN_PROGRESS
✓ Endpoint está sendo criado (isso é bom!)
NOT_READY
Ainda não pode receber requisições
READY
✓ Pronto para inferências

💡 Próximos Passos:
Aguarde 10-15 minutos - Este é o tempo normal de criação

Execute a Cell 39 periodicamente para verificar o status:

Se ainda estiver IN_PROGRESS: aguarde mais
Se estiver READY: você verá automaticamente o display(results_df) com os resultados!
Quando estiver READY, a Cell 39 executará automaticamente:

Fará 5 batches de inferências
Mostrará qual modelo foi usado (model-a ou model-b)
Exibirá o display(results_df) com as predições

⚠️ Importante:
NÃO execute a Cell 27 (delete endpoint) enquanto aguarda
NÃO execute a Cell 33 novamente - o endpoint já está sendo criado
APENAS execute a Cell 39 para verificar quando estiver pronto
Você pode monitorar também pela UI do Databricks em Machine Learning → Serving → M02-endpoint_default.

In [0]:
# Polling inteligente - verifica status a cada 30s por até 15 minutos
from time import sleep
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
max_wait_time = 15 * 60  # 15 minutos máximo
check_interval = 30  # Verifica a cada 30 segundos
elapsed = 0

print("⏳ Aguardando endpoint ficar READY...\n")

while elapsed < max_wait_time:
    try:
        endpoint = w.serving_endpoints.get('M02-endpoint_default')
        status = str(endpoint.state.ready)
        config_status = str(endpoint.state.config_update)
        
        print(f"[{elapsed//60}min {elapsed%60}s] Ready: {status}, Config: {config_status}")
        
        if status == "READY":
            print("\n🎉 ENDPOINT ESTÁ PRONTO!")
            break
        elif "FAILED" in config_status:
            print(f"\n❌ ERRO: Endpoint falhou na criação!")
            print("Execute Cell 27 para deletar e Cell 33 para recriar.")
            break
            
        sleep(check_interval)
        elapsed += check_interval
        
    except Exception as e:
        print(f"\n❌ Erro ao verificar status: {e}")
        break

if elapsed >= max_wait_time:
    print(f"\n⏰ Timeout após {max_wait_time//60} minutos. Verifique manualmente.")

⏳ Aguardando endpoint ficar READY...

in 0s] Ready: EndpointStateReady.NOT_READY, Config: EndpointStateConfigUpdate.UPDATE_FAILED

❌ ERRO: Endpoint falhou na criação!
Execute Cell 27 para deletar e Cell 33 para recriar.


In [0]:
# 🚀 BATCH INFERENCE SEM ENDPOINT - Funciona em qualquer tier!
# Esta é a alternativa quando Model Serving endpoints não estão disponíveis

import mlflow
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

print("📦 Carregando modelos diretamente do Unity Catalog...\n")

# Carregar Model A
model_a_uri = f"models:/{model_a_name}/{model_a_version}"
model_a = mlflow.pyfunc.load_model(model_a_uri)
print(f"✅ Model A carregado: {model_a_name}")

# Carregar Model B
model_b_uri = f"models:/{model_b_name}/{model_b_version}"
model_b = mlflow.pyfunc.load_model(model_b_uri)
print(f"✅ Model B carregado: {model_b_name}")

print("\n🔧 Preparando dados (encoding de variáveis categóricas)...\n")

# Preparar dados - encoding de variáveis categóricas
requests = X_large_test.copy().reset_index(drop=True)

# Encoder para gender: Female=0, Male=1, outros valores = 0
requests['gender'] = requests['gender'].map({'Female': 0, 'Male': 1}).fillna(0).astype('int64')

# Encoder para smoking_history
smoking_map = {
    'never': 0,
    'No Info': 1,
    'current': 2,
    'former': 3,
    'ever': 4,
    'not current': 5
}
requests['smoking_history'] = requests['smoking_history'].map(smoking_map).fillna(1).astype('int64')

print("✅ Dados preparados\n")
print("=" * 60)
print("\n🎯 Executando Batch Inference com A/B Testing simulado...\n")

# Configuração
results_df = pd.DataFrame(columns=["prediction", "model_served"])
batch_size = 10
number_of_batches = 5

# Simular split 50/50 entre os modelos
for batch_num in range(number_of_batches):
    # Pegar o batch
    batch_df = requests[requests.index // batch_size == batch_num]
    
    # Alternar entre Model A e Model B (50/50)
    if batch_num % 2 == 0:
        current_model = model_a
        model_name = "my-model-a"
    else:
        current_model = model_b
        model_name = "my-model-b"
    
    # Fazer predição
    predictions = current_model.predict(batch_df)
    
    print(f"Batch {batch_num + 1} response:")
    print(f"Model served: {model_name}")
    
    # Armazenar resultados
    batch_results = pd.DataFrame({
        "prediction": predictions,
        "model_served": model_name
    })
    
    results_df = pd.concat([results_df, batch_results], ignore_index=True)

print("\n" + "="*60)
print("\n✅ BATCH INFERENCE COMPLETO!\n")
print(f"Total de predições: {len(results_df)}")
print(f"\nDistribuição dos modelos:")
print(results_df['model_served'].value_counts())

# Exibir resultados
display(results_df)

📦 Carregando modelos diretamente do Unity Catalog...



✅ Model A carregado: workspace.default.my_model_16f17c6a


✅ Model B carregado: workspace.default.my_model_0ea6e9f3

🔧 Preparando dados (encoding de variáveis categóricas)...

✅ Dados preparados


🎯 Executando Batch Inference com A/B Testing simulado...

Batch 1 response:
Model served: my-model-a
Batch 2 response:
Model served: my-model-b
Batch 3 response:
Model served: my-model-a
Batch 4 response:
Model served: my-model-b
Batch 5 response:
Model served: my-model-a


✅ BATCH INFERENCE COMPLETO!

Total de predições: 50

Distribuição dos modelos:
model_served
my-model-a    30
my-model-b    20
Name: count, dtype: int64


prediction,model_served
0,my-model-a
0,my-model-a
0,my-model-a
0,my-model-a
0,my-model-a
0,my-model-a
0,my-model-a
0,my-model-a
0,my-model-a
0,my-model-a


In [0]:
# Verificar status do endpoint M02-endpoint_default
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

try:
    endpoint = w.serving_endpoints.get('M02-endpoint_default')
    print(f"✓ Endpoint 'M02-endpoint_default' existe!")
    print(f"\n📊 Status atual:")
    print(f"  Config Update: {endpoint.state.config_update}")
    print(f"  Ready: {endpoint.state.ready}")
    
    if str(endpoint.state.ready) == "READY":
        print("\n🎉 ENDPOINT ESTÁ PRONTO!")
        print("✓ Execute a Cell 39 para obter o display(results_df)!")
    else:
        print(f"\n⏳ Aguardando o endpoint ficar pronto...")
except Exception as e:
    print(f"❌ Erro: {e}")

✓ Endpoint 'M02-endpoint_default' existe!

📊 Status atual:
  Config Update: EndpointStateConfigUpdate.UPDATE_FAILED
  Ready: EndpointStateReady.NOT_READY

⏳ Aguardando o endpoint ficar pronto...


In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

endpoint = w.serving_endpoints.get('M02-endpoint_default')

print("🔍 DIAGNÓSTICO COMPLETO DO ENDPOINT:\n")
print(f"Estado Config Update: {endpoint.state.config_update}")
print(f"Estado Ready: {endpoint.state.ready}")

# Check for error messages
if hasattr(endpoint.state, 'message'):
    print(f"\n❌ Mensagem de Erro: {endpoint.state.message}")

# Check served entities
if hasattr(endpoint.config, 'served_entities'):
    print(f"\n📦 Entidades Configuradas:")
    for i, entity in enumerate(endpoint.config.served_entities, 1):
        print(f"\n  {i}. Nome: {entity.name}")
        print(f"     Entity Name: {entity.entity_name}")
        print(f"     Version: {entity.entity_version}")
        print(f"     Workload Size: {entity.workload_size}")

# Check pending config if exists
if hasattr(endpoint, 'pending_config') and endpoint.pending_config:
    print(f"\n⏳ Configuração Pendente detectada")

🔍 DIAGNÓSTICO COMPLETO DO ENDPOINT:

Estado Config Update: EndpointStateConfigUpdate.UPDATE_FAILED
Estado Ready: EndpointStateReady.NOT_READY

⏳ Configuração Pendente detectada


In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Get endpoint events
print("📋 EVENTOS DO ENDPOINT (últimos eventos):\n")
try:
    events = w.serving_endpoints.logs(name='M02-endpoint_default', served_model_name=None)
    for event in list(events)[:10]:  # Show last 10 events
        print(f"  - {event}")
except Exception as e:
    print(f"Não foi possível obter eventos: {e}")

print("\n" + "="*60)
print("\n💡 PRÓXIMOS PASSOS RECOMENDADOS:\n")
print("1️⃣  DELETAR o endpoint com falha:")
print("   workspace.serving_endpoints.delete(name='M02-endpoint_default')")
print("\n2️⃣  RECRIAR executando a Cell 33 novamente")
print("\n3️⃣  Possíveis causas de falha:")
print("   • Modelos não registrados corretamente no Unity Catalog")
print("   • Problemas de permissão")
print("   • Recursos indisponíveis no momento")

📋 EVENTOS DO ENDPOINT (últimos eventos):

Não foi possível obter eventos: Served entity with name 'None' does not exist with config version 0.


💡 PRÓXIMOS PASSOS RECOMENDADOS:

1️⃣  DELETAR o endpoint com falha:
   workspace.serving_endpoints.delete(name='M02-endpoint_default')

2️⃣  RECRIAR executando a Cell 33 novamente

3️⃣  Possíveis causas de falha:
   • Modelos não registrados corretamente no Unity Catalog
   • Problemas de permissão
   • Recursos indisponíveis no momento


## View Data about the Served Model

In addition to the frequency of the model served, we can view latency using the UI. Navigate to the model serving endpoint in **Serving** and scroll down to view **Metrics**. Here you will find Latency, Request Rate, Request Error, CPU Usage, Memory Usage, and Provisioned Concurrency. All of these should be taken into account when making a determination of rolling out a new model.

In the **Events** tab you will see the **Timestamp** along with **Event type**, **Served entity name**, and **message**. This can be useful for transparency in rolling out the model to determine how long it takes for any particular event.

Additionally, you can view the **Logs** of the serving model.

---

## Metric Tracking

Now that we have established our different models as well as setup our hypothesis, let's take a moment to establish which metrics we will be tracking. Since we are using a random forest classifier, some important metrics include F1-score, precision, recall, accuracy, and AUC-ROC. The table below summarizes the metric, its use case and focus area. We will not go into analysis here, as that is covered in a separate demonstration.

### Suggested Tracking Plan

| Metric | Use Case | Focus Area |
| :--- | :--- | :--- |
| **F1-Score** | Overall balance between precision and recall | Primary metric for A/B testing |
| **Precision and Recall** | Insights into specific model behaviors | For deeper performance insights |
| **AUC-ROC or PR AUC** | Discrimination ability and imbalance focus | Secondary evaluation |
| **Log Loss** | Prediction confidence | Confidence assessment |
| **Feature Importance** | New feature contribution | Validate feature utility |

## Additional Rollout Strategies

We close this demo with describing two additional common rollout strategies that we will go into due to time constraints. However, it is worth describing how Mosaic AI can still solve these problems.

---

## Adaptation to Blue-Green Rollout

Using Mosaic AI Model Serving makes it effortless to switch between different served models. Recall that **Blue-green testing** is a deployment strategy that minimizes the downtime to reduce the risk during the release of new features or updates to applications.

**Case 1: Zero Downtime**

If we need 0 downtime, we can simply have two endpoints deployed and expose the application to the correct API and immediately delete the old model serving endpoint. We would maintain both versions of the code but only delete the older one once the new is in use.

**Case 2: A Few Minutes Downtime**

If we are allowed to have a few minutes of downtime, we can use the same model serving endpoint and update the the version being served by using the following code. Note that this may take a few minutes to update, but we only have a single model serving endpoint deployed in this case.

To summarize, with zero downtime, we must maintain two separate model serving endpoints, which can be costly. If we allow for a few minutes of downtime, then we only have to maintain a single endpoint and simply perform the switch when we are ready. In the latter case, we don't need to maintain two endpoints or worry about cleaning up resources. With Mosaic AI Model Serving, the previous state of the endpoint is always running until the udpate to the new state is complete. This means the only element you have to adjust your rollout for is the time it takes to update to the new endpoint.

---

## Adaptation to Canary Rollout

Using Mosaic AI Model Serving, you can perform a gradual rollout by controlling the traffic directed to each model version. Unlike in A/B testing, where we might use a fixed 50/50 split to compare model versions, a canary rollout involves progressively increasing the traffic to the new model. For example, you could start with a 10/90 split (10% to the new model and 90% to the current model), then adjust to a 30/70 split, and so on, until the new model handles 100% of the traffic.

This gradual approach ensures the new model can be validated at scale without sacrificing uptime. Additionally, unlike the blue-green rollout strategy, a canary

## Additional Rollout Strategies

We close this demo with describing two additional common rollout strategies that we will go into due to time constraints. However, it is worth describing how Mosaic AI can still solve these problems.

---

## Adaptation to Blue-Green Rollout

Using Mosaic AI Model Serving makes it effortless to switch between different served models. Recall that **Blue-green testing** is a deployment strategy that minimizes the downtime to reduce the risk during the release of new features or updates to applications.

**Case 1: Zero Downtime**

If we need 0 downtime, we can simply have two endpoints deployed and expose the application to the correct API and immediately delete the old model serving endpoint. We would maintain both versions of the code but only delete the older one once the new is in use.

**Case 2: A Few Minutes Downtime**

If we are allowed to have a few minutes of downtime, we can use the same model serving endpoint and update the the version being served by using the following code. Note that this may take a few minutes to update, but we only have a single model serving endpoint deployed in this case.

To summarize, with zero downtime, we must maintain two separate model serving endpoints, which can be costly. If we allow for a few minutes of downtime, then we only have to maintain a single endpoint and simply perform the switch when we are ready. In the latter case, we don't need to maintain two endpoints or worry about cleaning up resources. With Mosaic AI Model Serving, the previous state of the endpoint is always running until the udpate to the new state is complete. This means the only element you have to adjust your rollout for is the time it takes to update to the new endpoint.

---

## Adaptation to Canary Rollout

Using Mosaic AI Model Serving, you can perform a gradual rollout by controlling the traffic directed to each model version. Unlike in A/B testing, where we might use a fixed 50/50 split to compare model versions, a canary rollout involves progressively increasing the traffic to the new model. For example, you could start with a 10/90 split (10% to the new model and 90% to the current model), then adjust to a 30/70 split, and so on, until the new model handles 100% of the traffic.

This gradual approach ensures the new model can be validated at scale without sacrificing uptime. Additionally, unlike the blue-green rollout strategy, a canary

## Conclusion

In this demonstration you learned about how to utilize MLflow along with Unity Catalog and Mosaic AI to enforce various rollout startegies with an emphasis on A/B testing. In addition, you learned how providing an alias can help keep track of which model you wish to rollout or test against. Finally, you learned how Mosaic AI allows for split traffic to make rollout strategies like blue-green and canary painless by simply updating the model serving endpoint.

© 2024 Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark and the Spark logo are trademarks of the Apache Software Foundation.

Privacy Policy | Terms of Use | Support

✅ Diagnóstico Completo:
Ambos os modelos estão READY! O problema NÃO é com os modelos, mas com a criação do endpoint em si.

📊 Respondendo sua pergunta original:
"Tem certeza que demora entre 10 e 15min?"

A resposta honesta é NÃO É GARANTIDO:

Cenário
Tempo
✅ Sucesso normal
5-15 minutos
⚠️ Recursos limitados
Pode levar mais tempo
❌ Falha imediata
0 segundos (como está acontecendo)
🐌 Conta free tier
Pode nem funcionar ou demorar muito mais
🔍 O problema atual:
O endpoint está falhando imediatamente porque provavelmente você está em um ambiente free tier ou com quota de recursos esgotada. O Databricks Free Tier geralmente:

❌ NÃO tem acesso a Model Serving endpoints
❌ Recursos computacionais muito limitados
❌ Não permite workload_size "Small" para serving
💡 Soluções possíveis:
1. Verificar se seu workspace tem Model Serving habilitado:

Machine Learning → Serving
Se não aparecer nada ou der erro de permissão, seu tier não suporta
2. Usar Batch Inference ao invés de Real-time Serving:

Carregar modelos diretamente com MLflow
Fazer predições em batch sem endpoint

![image_1787527337530.png](./image_1787527337530.png "image_1787527337530.png")

🔄 Quando usar cada abordagem:

Use Endpoint (Cell 39):

Aplicações em produção com requisições em tempo real
Precisa de alta disponibilidade 24/7
Tráfego distribuído automaticamente
Monitoramento de latência/throughput
Use Batch Inference (célula atual):
Processamento em lote (batch jobs)
Predições agendadas (ex: diariamente)
Ambiente de desenvolvimento/teste
Free tier ou recursos limitados
Análise exploratória

✅ Respondendo sua pergunta original:

"tem certeza que demora entre 10 e 15min o endpoint.state.ready) == 'READY'?"

Resposta: NÃO é garantido! E você descobriu isso na prática:

O endpoint falhou repetidamente
Provavelmente por limitações do tier/quota
Batch inference foi a solução correta para seu caso!


O endpoint falhou repetidamente
Provavelmente por limitações do tier/quota
Batch inference foi a solução correta para seu caso!